In [5]:
mep_ifc = r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\In\ifc\curated\51M-SKA-00-00-M3-MEP-0001_IFC 2x3-P01_with_port_relations.ifc"
arc_ifc = r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\In\ifc\raw\51M-SKA-00-00-A-ARC-0001_IFC 2x3-P01.ifc"
topology_ttl = r"C:\path\to\building_topology.ttl"
equipment_ttl = r"C:\path\to\equipment.ttl"

# Brick側でGUIDを格納している述語のIRIが分かっている場合は上書え
guid_pred_list = [
    "https://w3id.org/ifc/IFC2X3#globalId",
    # "http://example.com/ifc#guid",
]

In [7]:
# Notebookの最上流セルに配置
import sys
src_dir = r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# これでディレクトリ構造が src/topology_reasoner/reasoner.py なら次で解決
from topology_reasoner.reasoner import run_pairing

g, df, pairs = run_pairing(
    mep_ifc_path=r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\In\ifc\curated\51M-SKA-00-00-M3-MEP-0001_IFC 2x3-P01_with_port_relations.ifc",
    arc_ifc_path=r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\In\ifc\raw\51M-SKA-00-00-A-ARC-0001_IFC 2x3-P01.ifc",
    topology_ttl_path=r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\intermediate\building_topology.ttl",
    equipment_ttl_path=r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\intermediate\outputmorgate51_equipment.ttl",
    z_slack_m=0.2,
    guid_pred_iris=guid_pred_list,                    # or None
    write_ttl=True,
    out_ttl_path=r"data/Out/merged_with_locations.ttl",
    write_csv=True,
    report_csv_path=r"data/Out/pairing_report.csv",
)

In [9]:
from rdflib import Namespace
BRICK = Namespace("https://brickschema.org/schema/Brick#")
PROV = Namespace("http://www.w3.org/ns/prov#")

n_loc = sum(1 for _ in g.triples((None, BRICK.hasLocation, None)))
print("brick:hasLocation triples:", n_loc)

# 先頭20件の列挙
for i, (s, p, o) in enumerate(g.triples((None, BRICK.hasLocation, None))):
    if i >= 20:
        break
    print(f"{i+1:02d}. {s} -> {o}")

brick:hasLocation triples: 2297
01. https://id.morgate51.org/project#DSSO-00001 -> https://id.morgate51.org/project#03-01
02. https://id.morgate51.org/project#DSSO-00002 -> https://id.morgate51.org/project#03-01
03. https://id.morgate51.org/project#DSSO-00003 -> https://id.morgate51.org/project#03-01
04. https://id.morgate51.org/project#DSSO-00004 -> https://id.morgate51.org/project#03-01
05. https://id.morgate51.org/project#DSSO-00005 -> https://id.morgate51.org/project#03-01
06. https://id.morgate51.org/project#DSSO-00006 -> https://id.morgate51.org/project#03-01
07. https://id.morgate51.org/project#DSSO-00007 -> https://id.morgate51.org/project#03-01
08. https://id.morgate51.org/project#DSSO-00008 -> https://id.morgate51.org/project#03-01
09. https://id.morgate51.org/project#DSSO-00009 -> https://id.morgate51.org/project#03-01
10. https://id.morgate51.org/project#DSSO-00010 -> https://id.morgate51.org/project#03-01
11. https://id.morgate51.org/project#DSSO-00011 -> https://id.morgat

In [10]:
# 総トリプル数とクラス別件数の要約
print("triples:", len(g))

q = """
SELECT ?cls (COUNT(?s) AS ?n) WHERE {
  ?s a ?cls .
} GROUP BY ?cls ORDER BY ?cls
"""
for row in g.query(q):
    print(row)

triples: 31373
(rdflib.term.URIRef('https://brickschema.org/schema/Brick#Access_Control_System'), rdflib.term.Literal('27', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.URIRef('https://brickschema.org/schema/Brick#Air_Handling_Unit'), rdflib.term.Literal('3', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.URIRef('https://brickschema.org/schema/Brick#Air_Terminal_Unit'), rdflib.term.Literal('331', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.URIRef('https://brickschema.org/schema/Brick#Automatic_Transfer_Switch'), rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.URIRef('https://brickschema.org/schema/Brick#Boiler'), rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.URIRef('https://brickschema.org/schema/Brick#Booster_Pump'), rdflib.term.Liter

In [11]:
q = """
PREFIX bot: <https://w3id.org/bot#>
PREFIX m51: <https://id.morgate51.org/project#>
SELECT ?b ?s ?sp WHERE {
  ?b a bot:Building .
  ?b bot:hasStorey ?s .
  OPTIONAL { ?s bot:hasSpace ?sp }
}
ORDER BY ?b ?s ?sp
"""
res = g.query(q)
for row in list(res)[:30]:
    print(row)

(rdflib.term.URIRef('https://id.morgate51.org/project#51-Moorgate'), rdflib.term.URIRef('https://id.morgate51.org/project#B1'), rdflib.term.URIRef('https://id.morgate51.org/project#B1-01'))
(rdflib.term.URIRef('https://id.morgate51.org/project#51-Moorgate'), rdflib.term.URIRef('https://id.morgate51.org/project#B1'), rdflib.term.URIRef('https://id.morgate51.org/project#B1-02'))
(rdflib.term.URIRef('https://id.morgate51.org/project#51-Moorgate'), rdflib.term.URIRef('https://id.morgate51.org/project#B1'), rdflib.term.URIRef('https://id.morgate51.org/project#B1-03'))
(rdflib.term.URIRef('https://id.morgate51.org/project#51-Moorgate'), rdflib.term.URIRef('https://id.morgate51.org/project#B1'), rdflib.term.URIRef('https://id.morgate51.org/project#B1-04'))
(rdflib.term.URIRef('https://id.morgate51.org/project#51-Moorgate'), rdflib.term.URIRef('https://id.morgate51.org/project#B1'), rdflib.term.URIRef('https://id.morgate51.org/project#B1-05'))
(rdflib.term.URIRef('https://id.morgate51.org/proj

In [12]:
from pathlib import Path

out_path = Path(r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\intermediate\space_equipment.ttl")
out_path.parent.mkdir(parents=True, exist_ok=True)
g.serialize(destination=str(out_path), format="turtle")
print("written:", out_path.exists(), str(out_path))

written: True C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\intermediate\space_equipment.ttl
